In [1]:
import sys
import os
import time

# 自作パッケージを import
sys.path.append(os.path.abspath("../src"))
from sim_utils import *
import pymdp


In [2]:
# optimizerの準備
# モデル・データ読み込み
dof = 3
model = get_simple_arm(dof=dof)  # 自分のURDFを読み込むなら buildModelFromUrdf
data = model.createData()

velocity_scale = 5.0
effort_scale = 10.0

model.velocityLimit = np.full(dof, np.pi * velocity_scale)
model.effortLimit = np.full(dof, np.pi * effort_scale)
print(f"Angle Limits: {model.lowerPositionLimit}, {model.upperPositionLimit}")
print(f"Velocity Limits: -{model.velocityLimit}, {model.velocityLimit}")
print(f"Effort Limits: -{model.effortLimit}, {model.effortLimit}")

# エージェントの準備
bin_M = 30
eps = 1.0e-3 # 1.0e-3が先行研究
var_a = 1/bin_M 
var_b = 1/bin_M
num_iter = 16

num_obs = [bin_M for _ in range(dof*3)]
agent = RobotPerceptor(
    num_obs = num_obs,
    # num_obs=[10,10,10, 10,10,10, 10,10,10],
    # num_obs=[15, 15, 15, 15, 15, 15, 15, 15, 15],
    # num_obs=[bin_M, bin_M, bin_M, bin_M, bin_M, bin_M, bin_M, bin_M, bin_M],
    eps=eps,
    Avars=[var_a, var_a, var_a],
    Bvars=[var_b, var_b, var_b],
    num_iter=num_iter
)
# plot_A_all(agent.A)
# plot_B_all(agent.B)

fps = 30 # 本番は30
sec = 5 # 本番は5
time_steps = int(fps * sec) # 本番は150
dt = float(1/fps)
print(f"time_steps: {time_steps}, dt: {dt}")

# エンドエフェクタのframe名とid
frame_name = "ee_link"
frame_id = model.getFrameId(frame_name)
# 目標位置と姿勢の計算（SE3）
theta = np.pi/4
Rs = pin.utils.rotate("y", -3*theta)
start_placement = pin.SE3(Rs, np.array([-2, 0, 0]))
Re = pin.utils.rotate("y", 3*theta)
end_placement = pin.SE3(Re, np.array([2, 0, 0]))
start = ik_se3_solver(model, data, frame_id, start_placement)
end = ik_se3_solver(model, data, frame_id, end_placement)

# 関節角の制限
limits = same_limits(model.nq)
# 最適化のパラメータ
num_knots = 9
n_particles = 500
iters = 100


# Te = time.time()
# print(f"param set in ({Te-Ts:.2f}sec)")
Ts = time.time()

# Optimizerの初期化
opt = Optimizer(model, agent, time_steps, dt, start, end, limits, num_knots, n_particles)

Te = time.time()
print(f"optimizer initialized in {Te-Ts:.2f}sec")
# 事前分布となる動きの生成
# # 目的関数の最適化による生成
# best_cost, best_particle, best_qs = opt.optimize(
#     torque_change=1.0e-4, #~6.0e+4
#     jerk=1.0e-2, #~8.0e+3
#     iters=iters,
#     end_penalty=1.0e2, # ~3.0e+-1
#     temporal_approach_cost=1.0e1 # ~2.0e-1
#     )
Ts = time.time()


    
# 等速運動による生成
# best_particle, best_qs = opt.linear_base_particle()
# Te = time.time()
# print(f"const_beliefs generaeted in {Te-Ts:.2f}sec")
# Ts = time.time()
# const_beliefs = opt.agent.generate_const_beliefs(opt.env, best_qs, dt)
# opt.set_const_beliefs(best_qs)

# ジャーク、エネルギーなどの最小化による事前分布生成
jerk = 0.0#1.0e10
energy = 0.0
compensate_grav = False # True無重力と同じ、Falseなら姿勢維持に必要なエネルギーも含める
torque_change = 1.0

# result = opt.scipy_optimize(
#     jerk = jerk,
#     energy = energy,
#     torque_change = torque_change,
#     compensate_grav = compensate_grav,
#     maxiter=1.0e10,
#     ftol=1e-10,
#     initial_seed=1
#     )
_, linear_qs = opt.linear_base_particle()
opt.set_const_beliefs(linear_qs)
result = opt.pso_minimize(
    jerk = jerk,
    energy = energy,
    torque_change = torque_change,
    compensate_grav = compensate_grav
)

def format_weight(w):
    """重み w を指数表記または簡潔な文字列に変換する"""
    if w == 0.0:
        return "0"
    if w >= 1000 or w < 0.1:
        # 1000以上または0.1未満は指数表記 (例: 1e4, 1e-3)
        # ドットを回避し、ファイル名として安全な形式にする
        return f"{w:.0e}".replace(".", "p").replace("-", "m").replace("+", "")
    else:
        # それ以外は小数点第1位まで、ドットをアンダースコアに置換
        return f"{w:.1f}".replace(".", "p")
    
# 重力設定の文字列
grav_str = "GC" if compensate_grav else "GN"

# 命名規則の構築
param_str = (
    f"J{format_weight(jerk)}"
    f"_E{format_weight(energy)}"
    f"_TC{format_weight(torque_change)}"
    f"_{grav_str}"
)


Te = time.time()
print(f"const_beliefs set in {Te-Ts:.2f}sec")

savedir = "./tmp_movies/"
# movie = param_str
# movie = "energy_min_cgT"
# movie = "energy_min"
# movie = "jerk_min"
# plot_robot_motion(model, const_beliefs_qs, dt, savedir=savedir, movie=movie, detail=False, gripper_size=0.3, grid=False)
# plot_robot_motion(model, initial_qs, dt, detail=False, gripper_size=0.3, grid=False)
# plot_robot_motion(model, const_beliefs_qs, dt, detail=False, gripper_size=0.3, grid=False)


URDFファイルを ./robots/simple_arm_dof3.urdf に保存しました。自由度: 3
URDFファイルを ./robots/simple_arm_dof3.urdf に保存しました。自由度: 3
Angle Limits: [-3.14 -3.14 -3.14], [3.14 3.14 3.14]
Velocity Limits: -[15.70796327 15.70796327 15.70796327], [15.70796327 15.70796327 15.70796327]
Effort Limits: -[31.41592654 31.41592654 31.41592654], [31.41592654 31.41592654 31.41592654]


INFO:2025-12-11 09:28:17,846:jax._src.xla_bridge:752: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
2025-12-11 09:28:17,846 - jax._src.xla_bridge - INFO - Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


time_steps: 150, dt: 0.03333333333333333
収束しました！ iteration: 23
収束しました！ iteration: 23
optimizer initialized in 0.00sec


2025-12-11 09:29:49,756 - pyswarms.single.global_best - INFO - Optimize for 1 iters with {'c1': 1.5, 'c2': 1.5, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|1/1, best_cost=4.33e+4
2025-12-11 09:29:51,010 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 43295.01953125, best pos: [0.08024395 0.0270968  0.3033282  0.40083795 0.64636243 0.2075588
 0.64324454 0.1295494  0.79722681 0.39977496 0.36344136 0.41265043
 0.83843978 0.19738245 0.44286556 0.31230822 0.14455371 0.72085139
 0.02306171 0.70871699 0.96940502]


const_beliefs set in 92.14sec


In [5]:
plot_robot_motion(model, result.qs, dt )

2025-12-11 09:52:30,232 - matplotlib.animation - INFO - Animation.save using <class 'matplotlib.animation.HTMLWriter'>


Now preparing HTML display...


In [6]:
result.info()

           <<< Optimization Result Summary >>>          

[Optimization Result]
  Best Cost: 43295.019531
  Optimizer Type: GlobalBestPSO (Seed: None)

[Robot & Task Definition]
  Robot Model: simple_arm_dof3 (DOF: 3)
  Time: 150 steps (fps: 30 , dt: 0.0333 s, Total: 5.00 s)
  Start Q: [-0.32787016 -1.48490403 -0.54342002]...
  End Q: [0.32787016 1.48490403 0.54342002]...
  Limits Low/High (DOF 3): [Min: -3.14, Max: 3.14]

[Optimization Parameters]
  Max Iterations: 1
  Knot Points: 9
  Particles: 500
  Cost Weights:
    - torque_change  : 1.0000
 - Perception Agent: 使用 (16 反復)
   - DOF/Modality/Factors: 3 / 9 / 9
   - Params (eps, hist): 0.001 / 16
   - Avars/Bvars (Len): 9 / 9

[Final Computed Metrics]
  - jerk           : 2185.869286
  - energy         : 183.998811
  - torque_change  : 43295.018262

[Stored Data Statistics]
  Trajectory (qs): [Min: -1.4849, Max: 1.4849] (Shape: (150, 3))
  Best Particle: [Min: 0.0231, Max: 0.9694] (Shape: (21,))
  Cost History: 1 points (Initial: 43